# 03. PromptTemplate과 Message

PromptTemplate을 사용하면 프롬프트를 재사용 가능한 템플릿으로 관리할 수 있습니다.

In [5]:
import sys
sys.path.append("..")
from common_config import llm_connect
model = "gpt-4.1"
llm = llm_connect(model)

## 1. PromptTemplate - 단일 문자열 템플릿

In [6]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("{topic}에 대해 한 문장으로 설명해줘")

# 변수 채우기
prompt = template.invoke({"topic": "RAG"})
print(prompt)

text='RAG에 대해 한 문장으로 설명해줘'


## 2. ChatPromptTemplate - 역할 기반 템플릿

시스템/사용자/AI 메시지를 조합하여 템플릿을 만듭니다.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 전문가입니다."),
    ("human", "{question}")
])

messages = chat_prompt.invoke({
    "role": "파이썬",
    "question": "데코레이터가 뭔가요?"
})

print(messages)

messages=[SystemMessage(content='당신은 파이썬 전문가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='데코레이터가 뭔가요?', additional_kwargs={}, response_metadata={})]


## 3. LLM과 연결하여 호출

In [8]:
messages = chat_prompt.invoke({
    "role": "머신러닝",
    "question": "과적합이 뭔가요?"
})

response = llm.invoke(messages)
print(response.content)

과적합(Overfitting)이란 머신러닝이나 딥러닝 모델이 학습 데이터(training data)에 너무 지나치게 맞춰져서, 새로운 데이터(테스트 데이터나 실제 환경의 데이터)에는 잘 일반화하지 못하는 현상을 말합니다.

좀 더 쉽게 설명하면:

- 모델이 학습 데이터의 패턴뿐만 아니라, 그 안에 있는 **노이즈(잡음)**나 **특이한 예외**까지 모두 외워버리는 상태입니다.
- 이로 인해 학습 데이터에서는 성능(정확도 등)이 매우 높지만, 실제로는 새로운 데이터에 대해 예측력이 떨어집니다.

### 예시
- 수학 시험을 준비할 때, 문제집의 답만 외우고 원리를 이해하지 못한 경우와 비슷합니다. 문제집에 나온 문제는 잘 풀지만, 새로운 문제가 나오면 못 푸는 것과 같습니다.

### 과적합의 원인
- 모델이 너무 복잡할 때(파라미터가 많을 때)
- 학습 데이터가 너무 적을 때
- 학습을 너무 오래 시킬 때(에폭이 너무 많을 때)

### 과적합 방지 방법
- **데이터를 더 많이 모으기**
- **모델을 단순하게 만들기** (파라미터 수 줄이기)
- **정규화(Regularization) 기법 사용** (예: L1, L2 정규화)
- **드롭아웃(Dropout) 사용** (딥러닝에서)
- **조기 종료(Early Stopping) 사용** (검증 데이터의 성능이 더 이상 좋아지지 않을 때 학습 중단)

궁금한 점이 더 있으신가요?


## 4. 대화 히스토리 포함 - MessagesPlaceholder

이전 대화 내용을 프롬프트에 삽입할 때 사용합니다.

In [9]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

# 이전 대화 기록
history = [
    HumanMessage(content="내 이름은 홍길동이야"),
    AIMessage(content="안녕하세요, 홍길동님!")
]

messages = chat_prompt.invoke({
    "history": history,
    "question": "내 이름이 뭐라고?"
})

response = llm.invoke(messages)
print(response.content)

당신의 이름은 홍길동입니다!


In [11]:
messages

ChatPromptValue(messages=[SystemMessage(content='당신은 친절한 AI 어시스턴트입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='내 이름은 홍길동이야', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 홍길동님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={})])

In [10]:
# 현재 대화를 history에 추가
history.append(HumanMessage(content="내 이름이 뭐라고?"))
history.append(AIMessage(content=response.content))
history

[HumanMessage(content='내 이름은 홍길동이야', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 홍길동님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='당신의 이름은 홍길동입니다!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

## 5. 다중 변수 템플릿 예제

In [12]:
template = ChatPromptTemplate.from_messages([
    ("system", "당신은 {language} 번역가입니다."),
    ("human", "다음 문장을 번역해줘: {sentence}")
])

messages = template.invoke({
    "language": "영어",
    "sentence": "오늘 날씨가 정말 좋네요."
})

response = llm.invoke(messages)
print(response.content)

The weather is really nice today.


## 문제 해결

In [ ]:
template = ChatPromptTemplate.from_messages([
    ("system", """당신은 {sorted_language}에서 {target_language}로 번역하는 번역가입니다.
      출력을 할 때 {sorted_language}로 번역된 문장과 {target_language}로 번역된 문장을 모두 보여주세요.
     
      예시:
      입력(한국어): "오늘 날씨가 정말 좋네요.
      출력(영어): The weather is really nice today. 
     """),
    ("human", "다음 문장을 번역해줘: {text}")
])

messages = template.invoke({
    "sorted_language": "한국어",
    "target_language": "영어",
    "text": "오늘 날씨가 정말 좋네요."
})

response = llm.invoke(messages)
print(response.content)

입력(한국어): 나랑 맞짱뜰래?  
출력(영어): Do you want to fight me?
